In [1]:
%pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

In [ ]:
from ucimlrepo import fetch_ucirepo 

In [3]:
# fetch dataset 
diabetes_130_us_hospitals_for_years_1999_2008 = fetch_ucirepo(id=296) 
  
# data (as pandas dataframes) 
X = diabetes_130_us_hospitals_for_years_1999_2008.data.features 
y = diabetes_130_us_hospitals_for_years_1999_2008.data.targets 
  
# metadata 
print(diabetes_130_us_hospitals_for_years_1999_2008.metadata) 
  
# variable information 
print(diabetes_130_us_hospitals_for_years_1999_2008.variables) 

{'uci_id': 296, 'name': 'Diabetes 130-US Hospitals for Years 1999-2008', 'repository_url': 'https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008', 'data_url': 'https://archive.ics.uci.edu/static/public/296/data.csv', 'abstract': 'The dataset represents ten years (1999-2008) of clinical care at 130 US hospitals and integrated delivery networks. Each row concerns hospital records of patients diagnosed with diabetes, who underwent laboratory, medications, and stayed up to 14 days. The goal is to determine the early readmission of the patient within 30 days of discharge.\nThe problem is important for the following reasons. Despite high-quality evidence showing improved clinical outcomes for diabetic patients who receive various preventive and therapeutic interventions, many patients do not receive them. This can be partially attributed to arbitrary diabetes management in hospital environments, which fail to attend to glycemic control. Failure to provide pro

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


In [4]:
ids = diabetes_130_us_hospitals_for_years_1999_2008.data.ids
temp = ids.join(X)
df = temp.join(y)

In [ ]:
# save dataframe with original data
df.to_csv('original_data.csv')

In [5]:
print(df.columns)

Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='object')


In [6]:
print(df.groupby('weight').size())

weight
>200            3
[0-25)         48
[100-125)     625
[125-150)     145
[150-175)      35
[175-200)      11
[25-50)        97
[50-75)       897
[75-100)     1336
dtype: int64


In [7]:
# most common admitting medical specialties
df['medical_specialty'].value_counts(dropna=False)

medical_specialty
NaN                              49949
InternalMedicine                 14635
Emergency/Trauma                  7565
Family/GeneralPractice            7440
Cardiology                        5352
                                 ...  
SportsMedicine                       1
Speech                               1
Perinatology                         1
Neurophysiology                      1
Pediatrics-InfectiousDiseases        1
Name: count, Length: 73, dtype: int64

In [8]:
# find number of missing values for each column with missing values + overall percentage missing

# number of rows
n_rows = df.shape[0]
# column names 
columns = df.columns
col_percent_missing = []

# calculate percentage of missing values for each column 
for col in columns:
    col_na = df[col].isna().sum()
    col_percent = (col_na/n_rows)*100
    col_percent_missing.append((col, col_percent))

col_to_drop = []

# print columns with missing values
for col in col_percent_missing:
    if (col[1] > 0):
        print(f'{col[0]}: {col[1]:.2f}% missing')
        if (col[1] > 75):
            col_to_drop.append(col[0])


race: 2.23% missing
weight: 96.86% missing
payer_code: 39.56% missing
medical_specialty: 49.08% missing
diag_1: 0.02% missing
diag_2: 0.35% missing
diag_3: 1.40% missing
max_glu_serum: 94.75% missing
A1Cresult: 83.28% missing


In [9]:
# drop columns with > 75% missing values
df_new = df.drop(columns=col_to_drop)

In [10]:
# drop columns that are not needed for model prediction
drop_col = ['payer_code', 'num_lab_procedures']
df_temp = df_new.drop(columns=drop_col)

In [11]:
# remove rows with missing race and diagnosis information
df_updated = df_temp.dropna(subset = ['race', 'diag_1', 'diag_2', 'diag_3'])

In [12]:
# check for duplicate rows 
ids['encounter_id'].duplicated().any()

np.False_

In [13]:
# check for inconsistencies in data formatting (inconsistent capitalization, spacing, etc)
for col in df_updated.columns:
    print(df_updated[col].value_counts(dropna=False))

encounter_id
149190       1
191187564    1
191236368    1
191234376    1
191224722    1
            ..
108144486    1
108143628    1
108143292    1
108141846    1
443867222    1
Name: count, Length: 98053, dtype: int64
patient_nbr
88785891     39
1660293      23
88227540     23
23199021     23
23643405     22
             ..
24538824      1
4478517       1
23232438      1
29049372      1
175429310     1
Name: count, Length: 68630, dtype: int64
race
Caucasian          75079
AfricanAmerican    18881
Hispanic            1984
Other               1484
Asian                625
Name: count, dtype: int64
gender
Female             52833
Male               45219
Unknown/Invalid        1
Name: count, dtype: int64
age
[70-80)     25306
[60-70)     21809
[80-90)     16702
[50-60)     16697
[40-50)      9265
[30-40)      3548
[90-100)     2717
[20-30)      1478
[10-20)       466
[0-10)         65
Name: count, dtype: int64
admission_type_id
1    52178
3    18194
2    17543
6     5135
5     4661
8    

In [14]:
# remove one row with unknown gender value
df_updated.drop(df_updated[df_updated['gender'] == 'Unknown/Invalid'].index, inplace=True)

/var/folders/mh/40c656wd7r33_mswpgt_nng00000gn/T/ipykernel_71900/2722962567.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_updated.drop(df_updated[df_updated['gender'] == 'Unknown/Invalid'].index, inplace=True)


In [15]:
# remove null/unknown values from numerically mapped columns 

# admission - remove 5, 6, 8
df_updated = df_updated[~df_updated['admission_type_id'].isin([5, 6, 8])]

# discharge - remove 18, 25, 26
df_updated = df_updated[~df_updated['discharge_disposition_id'].isin([18, 25, 26])]

# admission source - remove 9, 15, 17, 20, 21  
df_updated = df_updated[~df_updated['admission_source_id'].isin([9, 15, 17, 20, 21])]

In [16]:
# change readmitted to binary values, where < 30 readmission is 1, otherwise 0

import numpy as np
df_updated['readmitted'] = np.where(df_updated['readmitted'] == '<30', 1, 0) 

In [17]:
df_updated['readmitted'].value_counts(dropna=False)

readmitted
0    73857
1     9397
Name: count, dtype: int64

In [18]:
# create new features 

# create col for total hospitalizations 
df_updated['num_hospitalizations'] = df_updated.groupby('patient_nbr')['patient_nbr'].transform('count')

# procedures/length of stay
df_updated['avg_procedure'] = df_updated['num_procedures'] / df_updated['time_in_hospital'] 

# number of total visits
df_updated['total_visits'] = df_updated['number_outpatient'] + df_updated['number_emergency'] + df_updated['number_inpatient']

# number of med changes 
df_updated['num_med_changes'] = df_updated.loc[:, 'metformin':'metformin-pioglitazone'].isin(['Down', 'Up']).sum(axis=1)

# number of med increases 
df_updated['num_med_increase'] = df_updated.loc[:, 'metformin':'metformin-pioglitazone'].isin(['Up']).sum(axis=1)

In [19]:
# drop columns not needed for model prediction

df_final = df_updated.drop(columns=df_updated.loc[:, 'metformin':'metformin-pioglitazone'].columns)
df_final = df_final.drop(columns=['medical_specialty', 'encounter_id', 'patient_nbr'])
df_final.columns

Index(['race', 'gender', 'age', 'admission_type_id',
       'discharge_disposition_id', 'admission_source_id', 'time_in_hospital',
       'num_procedures', 'num_medications', 'number_outpatient',
       'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3',
       'number_diagnoses', 'change', 'diabetesMed', 'readmitted',
       'num_hospitalizations', 'avg_procedure', 'total_visits',
       'num_med_changes', 'num_med_increase'],
      dtype='object')

In [20]:
# create boolean values for 'change' and 'diabetesMed' columns 
df_final['change'] = np.where(df_updated['change'] == 'Ch', 1, 0) 
df_final['diabetesMed'] = np.where(df_updated['diabetesMed'] == 'Yes', 1, 0) 

In [59]:
df_final.head(10)

,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_procedures,num_medications,number_outpatient,...,diag_3,number_diagnoses,change,diabetesMed,readmitted,num_hospitalizations,avg_procedure,total_visits,num_med_changes,num_med_increase
1,Caucasian,Female,[10-20),1,1,7,3,0,18,0,...,255,9,1,1,0,1,0.000000,0,1,1
2,AfricanAmerican,Female,[20-30),1,1,7,2,5,13,2,...,V27,6,0,1,0,1,2.500000,3,0,0
3,Caucasian,Male,[30-40),1,1,7,2,1,16,0,...,403,7,1,1,0,1,0.500000,0,1,1
4,Caucasian,Male,[40-50),1,1,7,1,0,8,0,...,250,5,1,1,0,1,0.000000,0,0,0
5,Caucasian,Male,[50-60),2,1,2,3,6,16,0,...,250,9,0,1,0,1,2.000000,0,0,0
6,Caucasian,Male,[60-70),3,1,2,4,1,21,0,...,V45,7,1,1,0,1,0.250000,0,0,0
7,Caucasian,Male,[70-80),1,1,7,5,0,12,0,...,250,8,0,1,0,1,0.000000,0,0,0
8,Caucasian,Female,[80-90),2,1,4,13,2,28,0,...,38,8,1,1,0,1,0.153846,0,0,0
9,Caucasian,Female,[90-100),3,3,4,12,3,18,0,...,486,8,1,1,0,1,0.250000,0,0,0
10,AfricanAmerican,Female,[40-50),1,1,7,9,2,17,0,...,996,9,0,1,0,1,0.222222,0,0,0


In [ ]:
# check outlier values

from scipy import stats 

df_outlier = df_final.copy()

# numeric columns
columns = ['time_in_hospital',
       'num_procedures', 'num_medications', 'number_outpatient',
       'number_emergency', 'number_inpatient',
       'number_diagnoses',
       'num_hospitalizations', 'avg_procedure', 'total_visits',
       'num_med_changes', 'num_med_increase']

# create new columns for z-scores
df_outlier[[col + "_zscore" for col in columns]] = stats.zscore(df_outlier[columns])

In [ ]:
# see distribution of outlier values
zscore_col = [col + "_zscore" for col in columns]
for col in zscore_col:
    print(df_outlier[col].value_counts())

In [60]:
# save processed dataframe 
df_final.to_csv('final_data.csv')

In [24]:
# load in icd code data
import json
with open('icd_codes.json') as file:
    icd_raw_data = json.load(file)

In [25]:
# test line to see how data is organized 
record = icd_raw_data[0]
print(record.get('children')[0].get('children')[0])

{'code': '00.0', 'desc': 'Therapeutic Ultrasound', 'children': [{'code': '00.01', 'desc': 'Therapeutic ultrasound of vessels of head and neck', 'children': []}, {'code': '00.02', 'desc': 'Therapeutic ultrasound of heart', 'children': []}, {'code': '00.03', 'desc': 'Therapeutic ultrasound of peripheral vascular vessels', 'children': []}, {'code': '00.09', 'desc': 'Other therapeutic ultrasound', 'children': []}]}


In [ ]:
# extract icd data from nested format

icd_data = []

# save code, primary category, subcategory, and diagnosis description
for category in icd_raw_data:
    primary_category = category.get('desc')
    sub_categories = category.get('children')
    for sub_category in sub_categories:
        sub_category_desc = sub_category.get('desc')
        codes = sub_category.get('children')
        for value in codes:
            code = value.get('code')
            code_desc = value.get('desc')
            icd_data.append({'code': code, 'description': code_desc, 'primary_category': primary_category, 'subcategory': sub_category_desc})

In [27]:
# save icd data to dataframe 
icd_code_df = pd.DataFrame(icd_data)

In [44]:
# export icd_dataframe 
icd_code_df.to_csv('icd_code.csv')